# Notebook 02 — Preprocessing & Feature Extraction

Demonstrates the full text cleaning pipeline and TF-IDF feature matrix.

In [ ]:
import sys; sys.path.insert(0, '..')
import pickle, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
import joblib
from src.preprocessing import clean_text, preprocess_text, preprocess_corpus
from src.features import build_tfidf, reduce_with_svd


## 1. Preprocessing Pipeline Walk-through

In [ ]:
sample = texts = [
    "The quick Brown Foxes are RUNNING over 42 lazy dogs! Visit http://example.com for more info.",
    "<b>NASA</b> launched a new satellite into low-earth orbit on Friday.",
    "The government's encryption policy involving NSA clipper chips was controversial.",
]
print("=== RAW TEXT ===")
for i, t in enumerate(sample): print(f"  [{i}] {t}")

print("\n=== AFTER clean_text() ===")
for i, t in enumerate(sample): print(f"  [{i}] {clean_text(t)}")

print("\n=== AFTER preprocess_text() (full pipeline) ===")
for i, t in enumerate(sample): print(f"  [{i}] {preprocess_text(t)}")


## 2. Vocabulary & Token Statistics

In [ ]:
from sklearn.datasets import fetch_20newsgroups
news = fetch_20newsgroups(subset='all', remove=('headers','footers','quotes'), random_state=42)
raw_texts = news.data[:500]  # sample for demo speed

processed, kept = preprocess_corpus(raw_texts, min_doc_len=10)
print(f"Input : {len(raw_texts)} docs")
print(f"Output: {len(processed)} docs ({len(raw_texts)-len(processed)} dropped)")

# Token count before vs after
import re, nltk
from nltk.corpus import stopwords
STOPS = set(stopwords.words('english'))
raw_tokens  = [re.findall(r'\b[a-z]{3,}\b', t.lower()) for t in raw_texts]
proc_tokens = [t.split() for t in processed]
raw_mean  = np.mean([len(t) for t in raw_tokens])
proc_mean = np.mean([len(t) for t in proc_tokens])
print(f"\nMean tokens — raw: {raw_mean:.1f}  |  processed: {proc_mean:.1f}")
print(f"Compression: {(1 - proc_mean/raw_mean)*100:.1f}% reduction")


In [ ]:
# Before / after token distribution plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist([len(t) for t in raw_tokens], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Token Count BEFORE Preprocessing'); axes[0].set_xlabel('Tokens')
axes[1].hist([len(t) for t in proc_tokens], bins=40, color='seagreen', edgecolor='white')
axes[1].set_title('Token Count AFTER Preprocessing'); axes[1].set_xlabel('Tokens')
plt.tight_layout(); plt.savefig('../outputs/figures/preprocessing_tokens.png', bbox_inches='tight'); plt.show()


## 3. TF-IDF Feature Matrix

In [ ]:
# Load pre-built TF-IDF for 20 Newsgroups
X_tfidf_ng = load_npz('../data/processed/X_tfidf_newsgroups.npz')
vec_ng = joblib.load('../models/tfidf_newsgroups.pkl')
print(f"TF-IDF shape: {X_tfidf_ng.shape}")
print(f"Sparsity    : {1 - X_tfidf_ng.nnz / np.prod(X_tfidf_ng.shape):.3%}")
print(f"Vocab size  : {len(vec_ng.vocabulary_):,}")
print(f"\nTop 30 terms by document frequency:")
df_freqs = (X_tfidf_ng > 0).sum(axis=0).A1
top_idx = df_freqs.argsort()[::-1][:30]
feat_names = vec_ng.get_feature_names_out()
print("  " + ", ".join(feat_names[top_idx]))


## 4. LSA — Explained Variance

In [ ]:
from sklearn.decomposition import TruncatedSVD
X_reduced_ng = np.load('../data/processed/X_reduced_newsgroups.npy')
print(f"Reduced matrix shape: {X_reduced_ng.shape}")

svd_ng = joblib.load('../models/svd_newsgroups.pkl')
explained = svd_ng.named_steps['svd'].explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(cumulative)+1), cumulative, marker='.', markersize=4, color='steelblue')
ax.axhline(0.5, linestyle='--', color='red', label='50% threshold')
ax.set_xlabel('Number of SVD Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('LSA — Explained Variance Ratio (20 Newsgroups)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('../outputs/figures/svd_variance_newsgroups.png', bbox_inches='tight'); plt.show()
print(f"100 components explain {cumulative[99]:.1%} of variance")
